In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

# This will create a default Spark session
spark = DataprocSparkSession.builder.getOrCreate()
# If you would like to customize the Spark session, please refer to the
# documentation at https://cloud.google.com/bigquery/docs/use-spark
#
# For example, if you need to use a different subnetwork, use the code below

# from google.cloud.dataproc_v1 import Session

# session = Session()
# session.environment_config.execution_config.subnetwork_uri = "<subnetwork_name>"
# spark = DataprocSparkSession.builder.dataprocSessionConfig(session).getOrCreate()

██████████████████████████▋                                                     

In [ ]:
from google.cloud import bigquery

# Inicializa o cliente do BigQuery padrão
client = bigquery.Client()

# Recupera 10% da base para uma exploração inicial
query = """
    SELECT *
    FROM `project-f1079166-4703-4cc3-975.raw_data.raw_people` TABLESAMPLE SYSTEM (10 PERCENT)
"""

# Traz para um DataFrame do Pandas para inspecionar de forma rápida
df_pandas = client.query(query).to_dataframe()
display(df_pandas.head())

,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,dia_sinistro,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito
0,2471524,29953,<NA>,3537800,PIEDADE,SOROCABA,ESTRADAS E RODOVIAS,None,NAO DISPONIVEL,None,...,9,2015/01,NaT,<NA>,<NA>,<NA>,None,None,PUBLICO,<NA>
1,2457183,1478,<NA>,3525409,JERIQUARA,FRANCA,VIAS URBANAS,None,NAO DISPONIVEL,None,...,10,2015/01,NaT,<NA>,<NA>,<NA>,None,None,PUBLICO,<NA>
2,2471347,26649,<NA>,3538808,PIRAJU,ITAPEVA,VIAS URBANAS,None,NAO DISPONIVEL,None,...,18,2015/01,NaT,<NA>,<NA>,<NA>,None,None,PUBLICO,<NA>
3,2470139,17523,<NA>,3532801,NOVA ALIANCA,SÃO JOSÉ DO RIO PRETO,ESTRADAS E RODOVIAS,None,NAO DISPONIVEL,None,...,31,2015/01,NaT,<NA>,<NA>,<NA>,None,None,PUBLICO,<NA>
4,2494681,77660,<NA>,3533403,NOVA ODESSA,CAMPINAS,ESTRADAS E RODOVIAS,None,NAO DISPONIVEL,None,...,18,2015/02,NaT,<NA>,<NA>,<NA>,None,None,PUBLICO,<NA>


In [ ]:
# Mostra o tipo de dado de cada coluna e se há valores nulos
print(df_pandas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122169 entries, 0 to 122168
Data columns (total 31 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   id_sinistro               122169 non-null  Int64 
 1   id_pessoa                 122169 non-null  Int64 
 2   id_veiculo                98345 non-null   Int64 
 3   cod_ibge                  122169 non-null  Int64 
 4   municipio                 122169 non-null  object
 5   regiao_administrativa     122169 non-null  object
 6   tipo_via                  122169 non-null  object
 7   tipo_veiculo_vitima       98487 non-null   object
 8   tipo_de_vitima            122169 non-null  object
 9   modo_transporte_vitima    99234 non-null   object
 10  sexo                      122169 non-null  object
 11  idade                     121056 non-null  Int64 
 12  gravidade_lesao           122169 non-null  object
 13  faixa_etaria_demografica  122169 non-null  object
 14  faix

**Resumo:**

* A base está dividida entre colunas numéricas de tipo inteiro (Int64), datas (dbdate) e colunas de texto/objeto (object).
* Campos como id_sinistro, id_pessoa, cod_ibge, municipio, regiao_administrativa, tipo_via, tipo_de_vitima, sexo, gravidade_lesao, faixa_etaria_demografica, faixa_etaria_legal e as variáveis temporais (data_sinistro, ano_sinistro, etc.) estão perfeitamente preenchidos, sem nulos;
* Campos como id_veiculo, tipo_veiculo_vitima, modo_transporte_vitima, profissao, grau_de_instrucao, nacionalidade, campos de obito possuem nulos que devem ser analisados e tratados.



## Avaliação de Variáveis Categóricas

In [ ]:
# Filtra apenas as colunas do tipo objeto (texto/categóricas)
colunas_categoricas = df_pandas.select_dtypes(include=['object']).columns

# Itera sobre cada coluna categórica mostrando a quantidade e os valores únicos
for coluna in colunas_categoricas:
    print(f"\n==================================================")
    print(f"COLUNA: {coluna}")
    print(f"Total de valores únicos: {df_pandas[coluna].nunique()}")
    print(f"Valores e frequências (Top 10):\n{df_pandas[coluna].value_counts().head(10)}")


COLUNA: municipio
Total de valores únicos: 644
Valores e frequências (Top 10):
municipio
SAO PAULO                29176
RIBEIRAO PRETO            3079
CAMPINAS                  3078
GUARULHOS                 2998
SAO JOSE DO RIO PRETO     2552
SAO BERNARDO DO CAMPO     2345
SOROCABA                  2210
SANTO ANDRE               1863
SANTOS                    1641
JUNDIAI                   1621
Name: count, dtype: int64

COLUNA: regiao_administrativa
Total de valores únicos: 16
Valores e frequências (Top 10):
regiao_administrativa
METROPOLITANA DE SÃO PAULO    47753
CAMPINAS                      18106
SOROCABA                       8214
SÃO JOSÉ DO RIO PRETO          7006
SÃO JOSÉ DOS CAMPOS            6328
BAIXADA SANTISTA               5930
RIBEIRÃO PRETO                 4788
MARÍLIA                        4058
BAURU                          3789
PRESIDENTE PRUDENTE            3699
Name: count, dtype: int64

COLUNA: tipo_via
Total de valores únicos: 3
Valores e frequências (Top 10)

**Resumo:**


* Criação da Variável Alvo (is_fatal) para o Machine Learning: O valor será 1 se gravidade_lesao == 'FATAL' ou se a data de óbito estiver preenchida; caso contrário, será 0.
* Padronização de Valores Nulos e Termos "NAO DISPONIVEL": Nos outputs, vemos que termos como "NAO DISPONIVEL" aparecem com frequência. Em frameworks de dados, strings de texto livre para ausência de informação atrapalham os modelos e os filtros do Power BI. Padronizar para null.
* Tratamento e Limpeza de Colunas Strings: Aplicar TRIM() e UPPERCASE em colunas de texto livre e remover acentuacao para evitar duplicidades por pequenas variações.
* Tratamento de Colunas com muitas categorias: a coluna de profissão possui muitos valores únicos possíveis, para o modelo de ML, categorias com pouca frequência geram ruído. Deverá ser avaliado isso.
* Grau de instrução possui muitos nulos, avaliar se deverá ser utilizado no modelo.

